# 🤖 Employee Attrition — ML Modeling

This notebook builds and evaluates classification models to predict
employee attrition. We follow a structured ML pipeline:
**Data Loading → Preprocessing → Baseline Model → Advanced Models → Evaluation**

In [2]:
# ── Data manipulation
import pandas as pd
import numpy as np

# ── Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# ── Preprocessing
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# ── Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# ── Evaluation metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, RocCurveDisplay
)

# ── Model saving
import joblib

# ── Settings
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries loaded successfully!")

✅ All libraries loaded successfully!


In [3]:
# Load dataset
df = pd.read_csv('../data/employee_performance_workload_attrition.csv')

# Quick check
print(f"Shape: {df.shape}")
print(f"\nColumns:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nAttrition distribution:\n{df['attrition'].value_counts()}")

Shape: (2800, 10)

Columns:
employee_id            int64
department            object
role_level            object
monthly_salary         int64
avg_weekly_hours       int64
projects_handled       int64
performance_rating     int64
absences_days          int64
job_satisfaction       int64
attrition             object
dtype: object

Missing values:
employee_id           0
department            0
role_level            0
monthly_salary        0
avg_weekly_hours      0
projects_handled      0
performance_rating    0
absences_days         0
job_satisfaction      0
attrition             0
dtype: int64

Attrition distribution:
attrition
No     1660
Yes    1140
Name: count, dtype: int64


## 📂 Data Loading

We reload the raw dataset fresh in this notebook to keep the
modeling pipeline **self-contained and reproducible** — independent
from the EDA notebook.

In [4]:
# ── Step 1: Encode target variable
df['attrition'] = df['attrition'].map({'Yes': 1, 'No': 0})

# ── Step 2: Encode categorical features with LabelEncoder
le = LabelEncoder()
df['department']  = le.fit_transform(df['department'])   # e.g. HR→2, Sales→4 ...
df['role_level']  = le.fit_transform(df['role_level'])   # Junior→0, Mid→1, Senior→2

# ── Step 3: Drop employee_id (not a predictive feature)
df.drop(columns=['employee_id'], inplace=True)

# ── Step 4: Define features (X) and target (y)
X = df.drop(columns=['attrition'])
y = df['attrition']

print(f"Features shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts()}")
print(f"\nAttrition rate: {y.mean()*100:.1f}%")

Features shape: (2800, 8)
Target distribution:
attrition
0    1660
1    1140
Name: count, dtype: int64

Attrition rate: 40.7%


## ⚙️ Preprocessing

| Step | Action | Reason |
|---|---|---|
| Target encoding | `Yes → 1`, `No → 0` | ML models require numeric target |
| Categorical encoding | `LabelEncoder` on `department` & `role_level` | Convert text to numbers |
| Drop `employee_id` | Removed from features | Arbitrary ID — not predictive |
| Feature/Target split | `X` = features, `y` = attrition | Standard ML convention |

> **Note:** No missing values were found in EDA, so no imputation is needed.
> No feature scaling required for tree-based models (Random Forest, XGBoost).
> Logistic Regression will handle unscaled data acceptably for baseline purposes.

## ✂️ Train / Test Split

- **80% training** / **20% test** split → standard ratio for this dataset size.
- `stratify=y` ensures the **attrition ratio is preserved** in both sets,
  preventing the model from training on an unrepresentative sample.
- `random_state=42` guarantees **reproducibility** — same split every run.

In [5]:
# Stratified split to preserve attrition ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,          # 80% train, 20% test
    random_state=42,        # reproducibility
    stratify=y              # keep Yes/No ratio balanced
)

print(f"Train size : {X_train.shape[0]} samples")
print(f"Test size  : {X_test.shape[0]} samples")
print(f"\nTrain attrition rate: {y_train.mean()*100:.1f}%")
print(f"Test  attrition rate: {y_test.mean()*100:.1f}%")

Train size : 2240 samples
Test size  : 560 samples

Train attrition rate: 40.7%
Test  attrition rate: 40.7%


Perfect! ✅ Everything looks great:

Split is correct — 2240 train / 560 test (80/20)

Stratification worked — both sets have exactly 40.7% attrition rate

Dataset is moderately imbalanced (40.7% vs 59.3%) — not extreme, so no need for SMOTE

## Logistic Regression (Baseline)

In [6]:
# ── Baseline model: simple, interpretable, fast
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

# Predictions
y_pred_lr = lr.predict(X_test)
y_prob_lr = lr.predict_proba(X_test)[:, 1]   # probability of attrition=1

print("── Logistic Regression ──")
print(f"Accuracy  : {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"Precision : {precision_score(y_test, y_pred_lr):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred_lr):.4f}")
print(f"F1 Score  : {f1_score(y_test, y_pred_lr):.4f}")
print(f"ROC-AUC   : {roc_auc_score(y_test, y_prob_lr):.4f}")

── Logistic Regression ──
Accuracy  : 0.6375
Precision : 0.5758
Recall    : 0.4167
F1 Score  : 0.4835
ROC-AUC   : 0.6842


## Random Forest

In [7]:
# ── Random Forest: ensemble of decision trees, handles non-linearity well
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# Predictions
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print("── Random Forest ──")
print(f"Accuracy  : {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"Precision : {precision_score(y_test, y_pred_rf):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred_rf):.4f}")
print(f"F1 Score  : {f1_score(y_test, y_pred_rf):.4f}")
print(f"ROC-AUC   : {roc_auc_score(y_test, y_prob_rf):.4f}")

── Random Forest ──
Accuracy  : 0.6304
Precision : 0.5607
Recall    : 0.4254
F1 Score  : 0.4838
ROC-AUC   : 0.6879


## XGBoost

In [8]:
# ── XGBoost: gradient boosting, typically best performance on tabular data
xgb = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=4,
    random_state=42,
    eval_metric='logloss',
    verbosity=0
)
xgb.fit(X_train, y_train)

# Predictions
y_pred_xgb = xgb.predict(X_test)
y_prob_xgb = xgb.predict_proba(X_test)[:, 1]

print("── XGBoost ──")
print(f"Accuracy  : {accuracy_score(y_test, y_pred_xgb):.4f}")
print(f"Precision : {precision_score(y_test, y_pred_xgb):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred_xgb):.4f}")
print(f"F1 Score  : {f1_score(y_test, y_pred_xgb):.4f}")
print(f"ROC-AUC   : {roc_auc_score(y_test, y_prob_xgb):.4f}")

── XGBoost ──
Accuracy  : 0.6393
Precision : 0.5691
Recall    : 0.4693
F1 Score  : 0.5144
ROC-AUC   : 0.6742


In [9]:
# Build comparison dataframe for all three models
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'Accuracy':  [0.6375, 0.6304, 0.6393],
    'Precision': [0.5758, 0.5607, 0.5691],
    'Recall':    [0.4167, 0.4254, 0.4693],
    'F1 Score':  [0.4835, 0.4838, 0.5144],
    'ROC-AUC':   [0.6842, 0.6879, 0.6742]
})

# Style the table — highlight max value in each column
results.style.highlight_max(
    subset=['Accuracy','Precision','Recall','F1 Score','ROC-AUC'],
    color='lightgreen'
).format(precision=4)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.6375,0.5758,0.4167,0.4835,0.6842
1,Random Forest,0.6304,0.5607,0.4254,0.4838,0.6879
2,XGBoost,0.6393,0.5691,0.4693,0.5144,0.6742


## 🏆 Model Comparison & Analysis

| Model | Accuracy | Precision | Recall | F1 Score | ROC-AUC |
|---|---|---|---|---|---|
| Logistic Regression | 0.6375 | **0.5758** | 0.4167 | 0.4835 | 0.6842 |
| Random Forest | 0.6304 | 0.5607 | 0.4254 | 0.4838 | **0.6879** |
| **XGBoost** | **0.6393** | 0.5691 | **0.4693** | **0.5144** | 0.6742 |

### 📊 Metric Breakdown

**Accuracy (~63–64%):**
- All models perform similarly — about 2 out of 3 predictions are correct.
- Accuracy alone is misleading with imbalanced data — focus on F1 & ROC-AUC.

**Recall (most important for HR use case):**
- **XGBoost (0.4693)** wins — it catches the most actual attrition cases.
- In HR, **missing a real attrition case (False Negative) is costly** — you
  lose an employee you could have retained. High recall is critical.

**F1 Score (balance of Precision & Recall):**
- **XGBoost (0.5144)** is the clear winner — best balance overall.

**ROC-AUC (~0.67–0.69):**
- All models are **moderately better than random (0.5)** but not exceptional.
- **Random Forest (0.6879)** edges out slightly on AUC.

### 🥇 Best Model: XGBoost
> XGBoost wins on **F1 Score** and **Recall** — the two most important
> metrics for an HR attrition use case where catching at-risk employees
> matters more than avoiding false alarms.

### 💡 Why scores are moderate (~0.63–0.64)?
> Our EDA revealed that most features have **very low correlation with
> attrition** (max -0.25). The dataset is synthetic with uniform
> distributions — making it genuinely hard to predict. This is expected
> and honest — a model claiming 95% accuracy on this data would be
> **overfitting**, not learning.